# We Convene MathSteps Agent — Math Practice Lab (Grades 6–12 + GED)

A Google Colab starter notebook for practicing several math task types with Python and SymPy. It provides guided, worked solutions; it is a learning aid, not an official Chicago Public Schools or GED product. Check current CPS and GED materials for official standards and test guidance.

**Safety:** The expression parser below accepts only numbers, basic arithmetic operators, parentheses, and the variables `x`, `y`, and `z`. It rejects function calls and other Python syntax. Do not enter student names or other personal information in a shared notebook.

## 1. Install the math library
Run this cell once per Colab runtime.

In [ ]:
%pip -q install sympy

## 2. Safe expression helpers
The parser validates Python's syntax tree before constructing a SymPy expression. Implicit multiplication (such as `2x`) is not accepted; write `2*x`. Use `**` for powers.

In [ ]:
import ast
import operator
from fractions import Fraction
import sympy as sp

x, y, z = sp.symbols('x y z')
_ALLOWED_NAMES = {'x': x, 'y': y, 'z': z}
_BINOPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.Pow: operator.pow}
_UNARYOPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def safe_expr(text):
    """Parse numeric arithmetic and x/y/z without eval or arbitrary calls."""
    if not isinstance(text, str) or not text.strip() or len(text) > 200:
        raise ValueError('Enter a non-empty expression of at most 200 characters.')
    try:
        tree = ast.parse(text.strip().replace('^', '**'), mode='eval')
    except SyntaxError as exc:
        raise ValueError('Check the expression syntax. Use * for multiplication and ** for powers.') from exc

    def visit(node):
        if isinstance(node, ast.Expression):
            return visit(node.body)
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            if abs(node.value) > 10**9:
                raise ValueError('Please use numbers with absolute value no greater than 1,000,000,000.')
            return sp.Rational(str(node.value))
        if isinstance(node, ast.Name) and node.id in _ALLOWED_NAMES:
            return _ALLOWED_NAMES[node.id]
        if isinstance(node, ast.BinOp) and type(node.op) in _BINOPS:
            left, right = visit(node.left), visit(node.right)
            if isinstance(node.op, ast.Pow):
                if not right.is_number or abs(float(right)) > 12:
                    raise ValueError('Use a numeric exponent between -12 and 12.')
            return _BINOPS[type(node.op)](left, right)
        if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARYOPS:
            return _UNARYOPS[type(node.op)](visit(node.operand))
        raise ValueError('Only numbers, x/y/z, parentheses, and + - * / ** are supported.')

    return sp.sympify(visit(tree))

def show_result(title, result, steps=None):
    print(f'We Convene MathSteps Agent | {title}')
    if steps:
        for number, step in enumerate(steps, 1):
            print(f'  Step {number}: {step}')
    print('  Answer:', result)


## 3. Task functions
Choose a task below. Examples are grouped by common grade-band topics and include GED-style foundational practice; grade bands are guides, not a standards certification.

In [ ]:
def simplify(expression):
    value = safe_expr(expression)
    return sp.simplify(value)

def solve_equation(equation, variable='x'):
    if equation.count('=') != 1:
        raise ValueError('Enter an equation with exactly one equals sign, such as 2*x + 3 = 11.')
    if variable not in _ALLOWED_NAMES:
        raise ValueError('Choose x, y, or z as the variable.')
    left_text, right_text = equation.split('=')
    left, right = safe_expr(left_text), safe_expr(right_text)
    target = _ALLOWED_NAMES[variable]
    steps = [f'Start with {sp.Eq(left, right)}']
    moved = sp.expand(left - right)
    if moved != left - right:
        steps.append(f'Combine and expand: {sp.Eq(moved, 0)}')
    solutions = sp.solve(sp.Eq(left, right), target)
    steps.append(f'Isolate {variable} using inverse operations and preserve equality.')
    return solutions, steps

def factor(expression):
    return sp.factor(safe_expr(expression))

def expand(expression):
    return sp.expand(safe_expr(expression))

def solve_system(equations, variables=('x', 'y')):
    if len(equations) != len(variables) or not equations:
        raise ValueError('Provide the same number of equations and variables (for example, two equations and x,y).')
    parsed = []
    for equation in equations:
        if equation.count('=') != 1:
            raise ValueError('Each equation must contain exactly one equals sign.')
        left, right = equation.split('=')
        parsed.append(sp.Eq(safe_expr(left), safe_expr(right)))
    targets = [_ALLOWED_NAMES[name] for name in variables]
    return sp.solve(parsed, targets, dict=True)

def proportion(a, b, c):
    """Solve a/b = c/d for d, using exact fractions where possible."""
    if b == 0 or a == 0:
        raise ValueError('The known numerator and denominator must be non-zero for this proportion helper.')
    answer = Fraction(b * c, a)
    return answer

def percent_of(percent, amount):
    return sp.Rational(str(percent)) * sp.Rational(str(amount)) / 100

def slope(x1, y1, x2, y2):
    if x2 == x1:
        return 'undefined (vertical line)'
    return sp.Rational(y2 - y1, x2 - x1)

def rectangle_area(length, width):
    if length < 0 or width < 0:
        raise ValueError('Length and width must be non-negative.')
    return length * width

def mean(values):
    if not values:
        raise ValueError('Enter at least one value.')
    return sp.Rational(sum(values), len(values))


## 4. Try examples across grade bands and GED prep
**Middle grades / GED foundations:** arithmetic, fractions, ratios, percentages, proportions, basic equations, area, and data.

**High school:** linear and quadratic equations, polynomial expansion/factoring, systems, and slope.

In [ ]:
show_result('Grades 6–8 / GED: order of operations', simplify('3 + 4 * (10 - 2)'))
show_result('Grades 6–8 / GED: fraction arithmetic', simplify('1/2 + 3/4'))
show_result('Grades 6–8 / GED: ratio / proportion, 3/4 = x/12', proportion(3, 4, 12))
show_result('Grades 6–8 / GED: 25 percent of 120', percent_of(25, 120))
solutions, steps = solve_equation('2*x + 3 = 11')
show_result('Grades 7–9 / GED: two-step equation', solutions, steps)
show_result('Geometry: area of a 8 by 5 rectangle', rectangle_area(8, 5))
show_result('Data: mean of 12, 15, 18, 21, 24', mean([12, 15, 18, 21, 24]))
show_result('Grades 9–12: expand', expand('2*(x + 3) + 5*x'))
show_result('Grades 9–12: factor a quadratic', factor('x**2 + 5*x + 6'))
solutions, steps = solve_equation('x**2 - 4*x + 3 = 0')
show_result('Grades 9–12: solve a quadratic', solutions, steps)
show_result('Coordinate geometry: slope through (2, 5) and (6, 13)', slope(2, 5, 6, 13))
show_result('Grades 9–12: solve a system', solve_system(['x + y = 10', 'x - y = 4']))

## 5. Try your own problems
Change the expression and run the cell. For an equation, use `solve_equation`; for a simplification, use `simplify`. Write multiplication explicitly (for example, `3*x`).

In [ ]:
# Try: simplify('4*x - 2*x + 5')
# Try: solve_equation('3*x - 7 = 11')
# Try: factor('x**2 - 9')
# Try: solve_system(['2*x + y = 7', 'x - y = 1'])
# Try: percent_of(15, 80)

## Scope and next steps
This notebook is an interactive practice prototype, not an autonomous AI tutor or a substitute for instruction. It does not yet include inequalities, geometry diagrams, word-problem interpretation, or all CPS/GED content. Verify curriculum claims and test-prep guidance against current official sources before classroom use. Do not upload student records or personally identifying information to a shared Colab notebook.

Notebook is versioned with the We Convene MathSteps Agent repository; run the cells in order after opening it in Colab.